# M6. 법령 조문 찾기

개인정보 보호법 현행판 조문을 조 · 항 · 호 단위 표로 DuckDB 에 넣고, LIKE 검색과 DuckDB 전문 검색(FTS · BM25)을 비교합니다. 형태소 분석기 없는 BM25 가 한국어 조사 · 띄어쓰기에서 어디서 틀리는지 숫자로 보고, 조사 떼기 · 2글자 n-gram 보정이 얼마나 도움이 되는지 잽니다.

- 단계: 기초 미니 프로젝트 · 비정형 · 예상 3~4시간
- 실행 환경: Google Colab(브라우저만 있으면 됩니다)
- 데이터: 법제처 국가법령정보 공동활용(DRF Open API) — 개인정보 보호법 현행판(법률 제21445호, 2026.3.10 공포 · 2026.9.11 시행) 원문, 2026-10-28 받은 스냅샷(P4 와 같은 파일). 법령은 저작권법 제7조에 따라 보호받지 않는 저작물

셀을 위에서부터 차례로 실행하세요. 각 단계 끝의 **확인 포인트** 숫자가 같게 나오면 다음 단계로 갑니다.
따라하기를 한 번 끝냈다면 같은 내용의 [빈칸 노트북](https://colab.research.google.com/github/aebonlee/materials/blob/main/data-eng/projects/M6/M6_blank.ipynb)으로 다시 쳐 보세요.

「유출」이라는 낱말이 들어간 조문을 찾는 가장 쉬운 방법은 `LIKE '%유출%'` 입니다. 그런데 찾은 수십 줄 가운데 **어느 것이 가장 관련 있는지**는 알려 주지 않습니다.
검색 엔진은 그래서 BM25 같은 점수로 순위를 매깁니다. DuckDB 에는 BM25 전문 검색(FTS) 확장이 들어 있어, 3주차에 쓰던 DuckDB 안에서 바로 해 볼 수 있습니다.

다만 BM25 는 원래 영어처럼 **띄어쓰기로 낱말이 나뉘는 언어**를 위해 만들어졌습니다. 한국어는 「개인정보가」 「개인정보를」 「개인정보의」가 모두 다른 낱말로 잡힙니다.
이 프로젝트에서는 그 한계를 일부러 드러내고, 형태소 분석기 없이 할 수 있는 보정 두 가지를 숫자로 비교합니다.

```text
법령 JSON → 조 · 항 · 호 표(pandas) → DuckDB 테이블 → LIKE 검색 → FTS 색인(BM25) → 조사 떼기 · 2글자 n-gram 색인 → 질문 15개로 정답률 비교
```

다 끝내면 다음 세 가지를 말할 수 있습니다.

- LIKE 와 BM25 가 각각 무엇을 잘하고 무엇을 못하는지
- 기본 FTS 색인이 한국어 법령에서 놓치는 것(조사 붙은 낱말 · 띄어쓰기 · 조문 번호의 숫자)
- 간단한 보정이 실제로 얼마나 도움이 되고, 어디서부터 임베딩 검색(P4)이 필요한지

## 1단계 · 준비 — DuckDB FTS 확장과 데이터

Colab 에는 DuckDB 1.x 가 깔려 있습니다. 전문 검색은 `fts` 라는 **확장(extension)** 으로 따로 들어 있어서, 처음 한 번 `INSTALL fts` 로 내려받고 `LOAD fts` 로 불러옵니다.
확장 파일은 DuckDB 판마다 따로 있으므로, 판이 바뀌면 다시 내려받습니다(`~/.duckdb/extensions/<판>/` 아래에 저장).

데이터는 P4(법령 Q&A 챗봇)와 같은 스냅샷입니다.

- `law_pipa_20260911.json.gz` — 개인정보 보호법 **현행판**(법률 제21445호, 2026.3.10 공포, 2026.9.11 시행). 2026-10-28 에 국가법령정보 공동활용 DRF API(`lawService.do?OC=test&target=law&type=JSON&MST=283839`)로 받았습니다.

법령은 저작권법 제7조(보호받지 못하는 저작물)에 해당해 자유롭게 이용할 수 있습니다. API 로 직접 받는 방법은 P4 의 2단계에 있습니다.

In [ ]:
import os, re, json, gzip, shutil, urllib.request
import pandas as pd
import duckdb

DATA = "https://raw.githubusercontent.com/aebonlee/materials/main/data-eng/projects/data/"

def fetch(name):
    """프로젝트 데이터를 ./data 폴더에 한 번만 받아 둡니다."""
    os.makedirs("data", exist_ok=True)
    dst = os.path.join("data", name)
    if not os.path.exists(dst):
        local = os.environ.get("DE_DATA_LOCAL")          # 오프라인 실습용 — 자료 폴더가 내 컴퓨터에 있을 때
        if local:
            shutil.copy(os.path.join(local, name), dst)
        else:
            urllib.request.urlretrieve(DATA + name, dst)
    return dst

path = fetch("law_pipa_20260911.json.gz")
print(path, round(os.path.getsize(path) / 1e3), "KB")

con = duckdb.connect()
con.sql("INSTALL fts")
con.sql("LOAD fts")
print("duckdb", duckdb.__version__)
print(con.sql("SELECT extension_name, installed, loaded FROM duckdb_extensions() WHERE extension_name = 'fts'").fetchall())

data/law_pipa_20260911.json.gz 52 KB
duckdb 1.3.2
[('fts', True, True)]


> **확인 포인트** — `[('fts', True, True)]` — 설치(installed)와 불러오기(loaded)가 둘 다 True 여야 합니다.

In [ ]:
assert con.sql("SELECT loaded FROM duckdb_extensions() WHERE extension_name = 'fts'").fetchall()[0][0]

> **자주 겪는 오류** — 회사 · 학교 망에서 `INSTALL fts` 가 `IO Error: Failed to download extension` 으로 실패하면 extensions.duckdb.org 로 나가는 연결이 막힌 것입니다.
> 다른 망에서 한 번 설치해 두거나, 확장 파일을 받아 `INSTALL '/경로/fts.duckdb_extension'` 으로 넣으세요.

## 2단계 · 조문을 표로 — 조 · 항 · 호 한 줄씩

법령 JSON 은 `조문단위` 목록 안에 `항` → `호` → `목` 이 겹겹이 들어 있습니다(P4 3단계). 검색하려면 이것을 **한 줄 = 항 하나 또는 호 하나**인 표로 펴야 합니다.
줄마다 조문 번호(`jo`) · 가지 번호(`br`, 「제7조의2」의 2) · 항 · 호를 따로 열로 두면, 검색 결과가 곧 「제○조제○항제○호」 출처가 됩니다.

In [ ]:
with gzip.open(path, "rt", encoding="utf-8") as f:
    law = json.load(f)["법령"]

CIRC = "①②③④⑤⑥⑦⑧⑨⑩⑪⑫⑬⑭⑮⑯⑰⑱⑲⑳"

def as_list(x):
    """항 · 호 · 목이 dict 하나로 오거나 없을 때도 목록으로 맞춥니다(P4 4단계)."""
    if x is None:
        return []
    return x if isinstance(x, list) else [x]

rows = []
for u in law["조문"]["조문단위"]:
    if u["조문여부"] != "조문":                                 # 「제1장 총칙」 같은 장 · 절 제목은 뺀다
        continue
    jo, br = int(u["조문번호"]), int(u.get("조문가지번호") or 0)
    title = u.get("조문제목", "")
    hangs = as_list(u.get("항"))
    if not hangs:                                               # 항 없이 본문만 있는 조문
        rows.append(dict(jo=jo, br=br, title=title, hang=0, ho="", text=u["조문내용"].strip()))
        continue
    for h in hangs:
        hn = (h.get("항번호") or "").strip()
        hang = CIRC.find(hn[:1]) + 1 if hn else 0
        if h.get("항내용"):
            rows.append(dict(jo=jo, br=br, title=title, hang=hang, ho="", text=h["항내용"].strip()))
        for ho in as_list(h.get("호")):
            hb = ho["호번호"].strip().rstrip(".") + (f"의{ho['호가지번호']}" if ho.get("호가지번호") else "")
            text = ho["호내용"].strip() + "".join("\n" + m["목내용"].strip() for m in as_list(ho.get("목")))
            rows.append(dict(jo=jo, br=br, title=title, hang=hang, ho=hb, text=text))

df = pd.DataFrame(rows)
print(df.shape)
df.head(5)

(856, 6)


,jo,br,title,hang,ho,text
0,1,0,목적,0,,제1조(목적) 이 법은 개인정보의 처리 및 보호에 관한 사항을 정함으로써 개인의 자...
1,2,0,정의,0,1,"1. ""개인정보""란 살아 있는 개인에 관한 정보로서 다음 각 목의 어느 하나에 해당..."
2,2,0,정의,0,1의2,"1의2. ""가명처리""란 개인정보의 일부를 삭제하거나 일부 또는 전부를 대체하는 등의..."
3,2,0,정의,0,2,"2. ""처리""란 개인정보의 수집, 생성, 연계, 연동, 기록, 저장, 보유, 가공,..."
4,2,0,정의,0,3,"3. ""정보주체""란 처리되는 정보에 의하여 알아볼 수 있는 사람으로서 그 정보의 주..."


이 표에는 검색을 방해하는 것이 둘 섞여 있습니다.

- `③ 삭제 <2020.2.4>` 처럼 **삭제된 항 · 조문**의 빈 자리
- 문장 끝의 `<개정 2014.3.24, 2015.7.24>` · `[본조신설 2023.3.14]` 같은 **개정 표시** — 「개정」이라는 낱말이 백 번 넘게 나와 검색 점수를 흐립니다

삭제 자리는 줄을 빼고, 개정 표시는 글에서 지웁니다. 조문 번호 글자(`제28조의2`)도 열로 만들어 둡니다.

In [ ]:
deleted = df["text"].str.match(r"^(\S+\s)?삭제\s*<")
law_df = df[~deleted].copy()
law_df["text"] = (law_df["text"]
                  .str.replace(r"\s*<(개정|신설|전문개정|본조신설)[^>]*>", "", regex=True)
                  .str.replace(r"\s*\[[^\]]*\]", "", regex=True)
                  .str.strip())
law_df["article"] = [f"제{j}조" + (f"의{b}" if b else "") for j, b in zip(law_df["jo"], law_df["br"])]
law_df.insert(0, "id", range(1, len(law_df) + 1))
print("삭제 자리:", deleted.sum(), "→ 남은 줄:", len(law_df))

삭제 자리: 18 → 남은 줄: 838


> **확인 포인트** — 조 · 항 · 호 856줄 중 삭제 자리 18줄을 빼고 **838줄**이 남습니다.

In [ ]:
assert len(df) == 856 and deleted.sum() == 18 and len(law_df) == 838

일부러 틀려 봅니다. 조문 번호를 **글자**로 정렬하면 어떻게 될까요?

In [ ]:
arts = law_df["article"].drop_duplicates()
print("글자 순서:", sorted(arts)[:6], "…")
print("제28조의10 이 제28조의2 보다 앞:", sorted(arts).index("제28조의10") < sorted(arts).index("제28조의2"))

글자 순서: ['제10조', '제11조', '제11조의2', '제12조', '제13조', '제13조의2'] …
제28조의10 이 제28조의2 보다 앞: True


> **자주 겪는 오류** — 글자로 비교하면 `제10조` 가 `제1조` 다음 · `제2조` 앞에 오고, `제28조의10` 이 `제28조의2` 보다 앞섭니다. 조문 순서는 반드시 숫자 열 `(jo, br)` 로 정렬하세요.
> 표에 번호를 숫자 열로 따로 둔 이유가 이것입니다.

## 3단계 · LIKE 검색 — 찾기는 하지만 순위가 없다

pandas 표를 DuckDB 테이블로 만듭니다. 3주차 14장처럼 파이썬 변수 이름을 그대로 SQL 에서 씁니다.

In [ ]:
con.sql("CREATE OR REPLACE TABLE law AS SELECT * FROM law_df")

like = con.sql("""
    SELECT article, hang, ho, left(text, 40) AS text
    FROM law
    WHERE text LIKE '%유출%'
""").df()
print(len(like), "줄")
like.head(6)

27 줄


,article,hang,ho,text
0,제23조,2,,② 개인정보처리자가 제1항 각 호에 따라 민감정보를 처리하는 경우에는 그
1,제24조,3,,③ 개인정보처리자가 제1항 각 호에 따라 고유식별정보를 처리하는 경우에는
2,제24조의2,2,,② 개인정보처리자는 제24조제3항에도 불구하고 주민등록번호가 유출등이 되
3,제25조,6,,⑥ 고정형영상정보처리기기운영자는 개인정보가 유출등이 되지 아니하도록 제2
4,제26조,4,,④ 위탁자는 업무 위탁으로 인하여 정보주체의 개인정보가 유출등이 되지 아
5,제28조의4,1,,① 개인정보처리자는 제28조의2 또는 제28조의3에 따라 가명정보를 처리


> **확인 포인트** — 「유출」이 들어간 줄은 **27줄**입니다. 결과는 표에 넣은 순서대로 나올 뿐, 어느 줄이 「유출」에 대한 조문인지 알려 주지 않습니다.

In [ ]:
assert len(like) == 27

LIKE 는 **글자 그대로** 비교합니다. 띄어쓰기 하나만 달라도 찾지 못합니다.

In [ ]:
for pat in ["%개인정보%", "%개인 정보%", "%개인정보 유출%"]:
    n = con.sql(f"SELECT count(*) FROM law WHERE text LIKE '{pat}'").fetchall()[0][0]
    print(f"{pat:14s} {n}줄")

%개인정보%         424줄
%개인 정보%        0줄
%개인정보 유출%      2줄


> **확인 포인트** — `%개인정보%` 는 424줄이지만, 띄어 쓴 `%개인 정보%` 는 **0줄**입니다. 질문하는 사람이 「개인 정보」라고 띄어 쓰면 LIKE 는 아무것도 못 찾습니다.

In [ ]:
assert con.sql("SELECT count(*) FROM law WHERE text LIKE '%개인 정보%'").fetchall()[0][0] == 0

## 4단계 · FTS 색인과 BM25 — 기본 설정 그대로

`PRAGMA create_fts_index('테이블', '번호 열', '글 열')` 이 색인을 만듭니다. 그러면 `fts_main_<테이블>` 이라는 스키마가 생기고, 그 안의 `match_bm25(번호, '질문')` 이 줄마다 점수를 돌려줍니다. 맞는 낱말이 하나도 없는 줄은 NULL 입니다.

BM25 점수는 세 가지로 정해집니다 — 질문 낱말이 그 줄에 **몇 번** 나오는가, 그 낱말이 전체에서 **얼마나 드문가**(드물수록 높게), 줄이 **얼마나 긴가**(길면 깎음).

In [ ]:
con.sql("PRAGMA create_fts_index('law', 'id', 'text')")

def search(table, q, k=5):
    return con.execute(f"""
        SELECT article, hang, ho, round(score, 2) AS score, left(text, 40) AS text FROM (
            SELECT *, fts_main_{table}.match_bm25(id, ?) AS score FROM {table}
        ) WHERE score IS NOT NULL
        ORDER BY score DESC LIMIT {k}
    """, [q]).df()

search("law", "유출")

,article,hang,ho,score,text
0,제31조,4,6,3.30,6. 개인정보 유출 및 오용ㆍ남용 방지를 위한 내부통제시스템의 구축
1,제24조의2,2,,1.75,② 개인정보처리자는 제24조제3항에도 불구하고 주민등록번호가 유출등이 되


순위는 생겼는데, 27줄이 아니라 **몇 줄만** 나옵니다. 색인이 글을 어떻게 낱말로 나눴는지 `tokenize` 로 들여다봅니다.

In [ ]:
print(con.sql("SELECT fts_main_law.tokenize('제34조제1항에 따라 개인정보가 유출되면')").fetchall()[0][0])
n_default = len(search("law", "유출", k=100))
print("기본 색인으로 「유출」이 맞는 줄:", n_default)

['제', '조제', '항에', '따라', '개인정보가', '유출되면']
기본 색인으로 「유출」이 맞는 줄: 2


> **확인 포인트** — 낱말이 `['제', '조제', '항에', '따라', '개인정보가', '유출되면']` 로 나뉩니다. 「유출」은 띄어 쓴 「유출」 그대로인 **2줄**만 맞습니다 — 「유출되면」 「유출등이」 「유출된」은 모두 다른 낱말입니다.

In [ ]:
assert n_default == 2

여기서 보이는 문제가 셋입니다.

- **조사 · 어미**: 「유출되면」은 「유출」과 다른 낱말입니다. 영어용 기본 설정(`stemmer='porter'`)은 영어 어미(-ing, -ed)만 떼고, 한국어 조사는 건드리지 않습니다.
- **숫자가 사라짐**: 기본 설정은 영문 소문자가 아닌 글자 중 숫자 · 기호를 지웁니다(`ignore='(\\.|[^a-z])+'`). 「제34조제1항」이 `제 · 조제 · 항` 이 되어 조문 번호로 찾을 수 없습니다.
- **띄어쓰기**: 「개인 정보」로 물으면 `개인` · `정보` 두 낱말로 나뉘어, 「개인정보」와 맞지 않습니다.

일부러 조문 번호로 찾아봅니다.

In [ ]:
try:
    hit = search("law", "제34조")
    print(hit)
    assert hit.iloc[0]["article"] == "제34조", "1위가 제34조가 아닙니다"
except AssertionError as e:
    print("실패:", e, "— 숫자가 지워져 「제」 「조」만으로 찾았기 때문입니다")

  article  hang ho  score                                      text
0    제26조     8      3.43  ⑧ 수탁자에 관하여는 제15조부터 제18조까지, 제21조, 제22조, 제
1  제28조의7     0      2.98  제28조의7(적용범위) 제28조의2 또는 제28조의3에 따라 처리된 가명
2  제35조의2     4  2   2.90                          2. 「지방세기본법」 제86조
3    제58조     2      2.72  ② 제25조제1항 각 호에 따라 공개된 장소에 고정형 영상정보처리기기를 
4    제58조     3      2.51  ③ 개인정보처리자가 동창회, 동호회 등 친목 도모를 위한 단체를 운영하기
실패: 1위가 제34조가 아닙니다 — 숫자가 지워져 「제」 「조」만으로 찾았기 때문입니다


> **Aebon's Tip** — 조문 번호처럼 **정확히 맞아야 하는 값**은 전문 검색이 아니라 `WHERE article = '제34조'` 같은 열 조건으로 찾는 것이 맞습니다. 2단계에서 번호를 열로 따로 둔 것이 여기서 쓰입니다.

## 5단계 · 보정 두 가지 — 조사 떼기와 2글자 n-gram

형태소 분석기(Kiwi · Mecab 등) 없이 할 수 있는 보정 두 가지를 만듭니다. 질문과 글에 **똑같은 처리**를 해야 맞습니다.

- **조사 떼기** — 낱말 끝의 흔한 조사(을 · 를 · 이 · 가 · 은 · 는 · 의 · 에 · 에서 · 으로 …)를 뗍니다. 「개인정보가」 → 「개인정보」. 남은 낱말이 두 글자는 되게 해 「사이」 같은 낱말이 망가지지 않게 합니다.
- **2글자 n-gram** — 공백을 모두 없애고 두 글자씩 겹쳐 자릅니다. 「유출되면」 → `유출 · 출되 · 되면`. 띄어쓰기와 조사에 상관없이 「유출」이 들어 있으면 맞습니다.

In [ ]:
JOSA = sorted(["에서는", "에게는", "으로는", "에서", "에게", "으로", "까지", "부터", "이나",
               "을", "를", "이", "가", "은", "는", "의", "에", "로", "와", "과", "도", "만"], key=len, reverse=True)

def words(t):
    return re.findall(r"[가-힣a-zA-Z0-9]+", t)

def strip_josa(t):
    out = []
    for w in words(t):
        for j in JOSA:                                       # 긴 조사부터 — 「에서」를 「에」보다 먼저
            if len(w) - len(j) >= 2 and w.endswith(j):
                w = w[:-len(j)]
                break
        out.append(w)
    return " ".join(out)

def bigram(t):
    s = "".join(words(t))                                    # 공백을 없앤다 — 띄어쓰기 차이가 사라진다
    return " ".join(s[i:i + 2] for i in range(len(s) - 1))

print(strip_josa("개인정보가 유출되면 정보주체에게 알려야 하나요"))
print(bigram("개인 정보 유출"))

개인정보 유출되면 정보주체 알려야 하나요
개인 인정 정보 보유 유출


보정한 글을 열로 만들어 테이블을 하나씩 더 만들고 색인합니다. 이번에는 숫자와 한글을 지우지 않도록 `ignore` 를 바꾸고, 영어용 어간 처리와 불용어는 끕니다.
DuckDB FTS 색인은 **테이블 하나에 하나**라서, 보정마다 테이블을 따로 둡니다.

In [ ]:
law_df["t_josa"] = law_df["text"].map(strip_josa)
law_df["t_bigram"] = law_df["text"].map(bigram)

for name, col in [("law_josa", "t_josa"), ("law_bigram", "t_bigram")]:
    con.sql(f"CREATE OR REPLACE TABLE {name} AS SELECT id, article, hang, ho, text, {col} AS body FROM law_df")
    con.sql(f"""PRAGMA create_fts_index('{name}', 'id', 'body',
                stemmer='none', stopwords='none', ignore='(\\.|[^a-z0-9가-힣])+')""")

prep = {"law": lambda q: q, "law_josa": strip_josa, "law_bigram": bigram}
def search2(table, q, k=5):
    return search(table, prep[table](q), k)

search2("law_bigram", "유출")

,article,hang,ho,score,text
0,제64조의2,5,5,2.37,5. 개인정보가 유출등이 된 경우 위반행위와의 관련성 및 유출등의 규모
1,제34조,1,2,2.20,2. 유출등이 된 시점과 그 경위
2,제34조,1,1,2.18,1. 유출등이 된 개인정보의 항목
3,제31조,4,6,1.89,6. 개인정보 유출 및 오용ㆍ남용 방지를 위한 내부통제시스템의 구축
4,제34조,2,,1.86,"② 제1항에도 불구하고 개인정보의 유형, 정보주체에게 미치는 영향 및 유"


이제 낱말 몇 개로 **「LIKE 로 찾은 줄 수」**와 색인마다 맞은 줄 수를 나란히 놓습니다. LIKE 는 순위는 없지만 글자가 들어 있는 줄은 빠짐없이 찾으므로, 놓친 줄을 세는 기준으로 씁니다.
여러 낱말이 든 질문은 모든 낱말이 맞는 줄만 세도록 `conjunctive := 1` 을 줍니다.

In [ ]:
def n_match(table, q):
    return con.execute(f"""
        SELECT count(*) FROM (SELECT fts_main_{table}.match_bm25(id, ?, conjunctive := 1) AS s FROM {table})
        WHERE s IS NOT NULL
    """, [prep[table](q)]).fetchall()[0][0]

recall = []
for kw in ["유출", "파기", "동의", "가명정보", "보호책임자", "개인 정보"]:
    n_like = con.sql(f"SELECT count(*) FROM law WHERE text LIKE '%{kw}%'").fetchall()[0][0]
    recall.append({"word": kw, "like": n_like, **{t: n_match(t, kw) for t in prep}})
recall = pd.DataFrame(recall)
recall

,word,like,law,law_josa,law_bigram
0,유출,27,2,4,27
1,파기,14,4,5,14
2,동의,60,16,52,60
3,가명정보,15,2,15,15
4,보호책임자,15,3,15,15
5,개인 정보,0,0,8,424


> **확인 포인트** — 「유출」은 LIKE 27줄 · 기본 색인 2줄 · 조사 떼기 4줄 · 2-gram 27줄, 「가명정보」는 15 · 2 · 15 · 15줄입니다.
> 조사 떼기는 「가명정보를」 같은 **명사 + 조사**는 살리지만 「유출되면」 「유출된」 같은 **동사 활용**은 못 살립니다. 2-gram 은 LIKE 와 같은 수를 찾고, 띄어 쓴 「개인 정보」로도 「개인정보」가 든 424줄을 모두 찾습니다.

In [ ]:
r = recall.set_index("word")
assert r.loc["유출"].tolist() == [27, 2, 4, 27]
assert r.loc["가명정보"].tolist() == [15, 2, 15, 15]
assert r.loc["개인 정보", "like"] == 0 and r.loc["개인 정보", "law_bigram"] == 424

그런데 2-gram 이 많이 찾는 것이 늘 좋은 일은 아닙니다. 2-gram 은 **낱말 경계를 모릅니다.** 「위탁자」를 `위탁 · 탁자` 로 쪼개 찾으므로, 「위탁」과 「수탁자」가 한 줄에 따로 있어도 맞습니다.

In [ ]:
n_like = con.sql("SELECT count(*) FROM law WHERE text LIKE '%위탁자%'").fetchall()[0][0]
n_bg = n_match("law_bigram", "위탁자")
print("LIKE:", n_like, "줄  /  2-gram:", n_bg, "줄")
extra = con.execute("""
    SELECT article, hang, left(text, 60) AS text FROM (
        SELECT *, fts_main_law_bigram.match_bm25(id, ?, conjunctive := 1) AS s FROM law_bigram
    ) WHERE s IS NOT NULL AND text NOT LIKE '%위탁자%'
""", [bigram("위탁자")]).df()
extra

LIKE: 5 줄  /  2-gram: 8 줄


,article,hang,text
0,제26조,5,⑤ 수탁자는 개인정보처리자로부터 위탁받은 해당 업무 범위를 초과하여 개인정보를 이용...
1,제26조,7,⑦ 수탁자가 위탁받은 업무와 관련하여 개인정보를 처리하는 과정에서 이 법을 위반하여...
2,제75조,4,5. 제26조제2항을 위반하여 위탁하는 업무의 내용과 수탁자를 공개하지 아니한 자


> **확인 포인트** — LIKE 는 5줄, 2-gram 은 **8줄**입니다. 더 찾은 3줄은 「위탁자」가 아니라 「위탁받은 · 위탁하는」과 「수탁자」가 따로 든 줄 — **엉뚱한 일치(거짓 양성)**입니다.

In [ ]:
assert (n_like, n_bg, len(extra)) == (5, 8, 3)

> **자주 겪는 오류** — 띄어쓰기를 무시하는 대가로 거짓 양성이 늘어납니다. 그래서 2-gram 은 낱말 하나를 정확히 찾는 데보다, 질문 전체의 2-gram 점수를 더해 **순위**를 매길 때 힘을 냅니다. 낱말 하나를 빠짐없이 찾는 일은 여전히 LIKE 가 정확합니다.

## 6단계 · 질문 15개로 정답률 재기

실제 질문처럼 조사와 어미가 붙은 문장 15개를 만들고, 정답 조문을 정해 둡니다. 색인마다 1위 조문이 정답인지(hit@1), 3위 안에 있는지(hit@3)를 셉니다.
줄(항 · 호) 단위로 검색하지만, 정답은 조문 단위로 비교합니다.

In [ ]:
gold = {
    "개인정보가 유출되면 정보주체에게 알려야 하나요?": "제34조",
    "가명정보를 처리할 수 있는 목적은?": "제28조의2",
    "만 14세 미만 아동의 개인정보를 처리하려면?": "제22조의2",
    "개인정보를 파기해야 하는 때는?": "제21조",
    "민감정보는 어떤 경우에 처리할 수 있나요?": "제23조",
    "공공장소에 영상정보처리기기를 설치해도 되나요?": "제25조",
    "개인정보 보호책임자는 누가 지정하나요?": "제31조",
    "동의 없이 제3자에게 제공할 수 있나요?": "제17조",
    "개인정보를 수집할 수 있는 경우는?": "제15조",
    "주민등록번호를 처리할 수 있나요?": "제24조의2",
    "손해배상을 청구할 수 있나요?": "제39조",
    "개인정보 처리방침을 공개해야 하나요?": "제30조",
    "정보주체가 자기 정보의 열람을 요구하려면?": "제35조",
    "개인정보 분쟁조정위원회는 무엇을 하나요?": "제40조",
    "개인정보를 국외로 이전할 수 있나요?": "제28조의8",
}

rows = []
for q, ans in gold.items():
    row = {"question": q[:18], "gold": ans}
    for t in prep:
        top = search2(t, q, k=3)["article"].tolist()
        row[t] = top[0] if top else "-"
        row[t + "_hit1"] = bool(top) and top[0] == ans
        row[t + "_hit3"] = ans in top
    rows.append(row)
evalq = pd.DataFrame(rows)
print(evalq[["question", "gold", "law", "law_josa", "law_bigram"]])
score = pd.DataFrame({t: [evalq[t + "_hit1"].sum(), evalq[t + "_hit3"].sum()] for t in prep}, index=["hit@1", "hit@3"])
score

              question    gold     law law_josa law_bigram
0   개인정보가 유출되면 정보주체에게     제34조    제75조     제20조       제36조
1   가명정보를 처리할 수 있는 목적은  제28조의2  제28조의2   제28조의2     제28조의2
2   만 14세 미만 아동의 개인정보를  제22조의2    제38조     제38조       제38조
3    개인정보를 파기해야 하는 때는?    제21조    제30조     제30조       제21조
4   민감정보는 어떤 경우에 처리할 수    제23조    제40조   제24조의2       제23조
5   공공장소에 영상정보처리기기를 설치    제25조    제25조     제25조       제25조
6   개인정보 보호책임자는 누가 지정하    제31조    제31조     제31조       제31조
7   동의 없이 제3자에게 제공할 수     제17조    제17조     제17조       제27조
8   개인정보를 수집할 수 있는 경우는    제15조    제15조     제15조       제15조
9   주민등록번호를 처리할 수 있나요?  제24조의2  제24조의2   제24조의2     제24조의2
10    손해배상을 청구할 수 있나요?    제39조    제39조     제39조     제39조의2
11  개인정보 처리방침을 공개해야 하나    제30조  제30조의2   제30조의2        제3조
12  정보주체가 자기 정보의 열람을 요    제35조    제35조     제35조       제35조
13  개인정보 분쟁조정위원회는 무엇을     제40조  제50조의2     제40조       제40조
14  개인정보를 국외로 이전할 수 있나  제28조의8  제28조의8   제28조의8     제28조의8


,law,law_josa,law_bigram
hit@1,9,10,10
hit@3,14,14,13


> **확인 포인트** — hit@1 은 기본 9 · 조사 떼기 10 · 2-gram 10, hit@3 은 14 · 14 · 13 (15문항 중)입니다.
> 보정이 한두 문항씩은 건지지만, 「유출되면 알려야」 질문은 세 색인 모두 제34조를 1위로 올리지 못합니다 — 질문에 「통지」라는 법령 낱말이 없기 때문입니다.

In [ ]:
assert score.loc["hit@1"].tolist() == [9, 10, 10]
assert score.loc["hit@3"].tolist() == [14, 14, 13]
assert not evalq.loc[0, ["law_hit1", "law_josa_hit1", "law_bigram_hit1"]].any()

BM25 는 **같은 글자**가 있어야 점수를 줍니다. 「알려야」와 「통지하여야」, 「AI」와 「인공지능」처럼 뜻은 같고 글자가 다른 말은 어떤 보정으로도 이어지지 않습니다.
뜻으로 찾으려면 문장을 벡터로 바꾸는 임베딩 검색이 필요합니다 — P4(법령 Q&A 챗봇)에서 같은 법령을 ko-sroberta 임베딩 + Qdrant 로 찾으면 유출 질문의 1위가 제34조제1항이 됩니다.

> **Aebon's Tip** — 실무에서는 BM25 와 임베딩 검색을 **함께** 씁니다(하이브리드 검색). 조문 번호 · 고유명사(「주민등록번호」)처럼 글자가 딱 맞아야 하는 질문은 BM25 가, 말을 바꿔 묻는 질문은 임베딩이 잘 찾기 때문입니다. 두 점수를 순위로 합치는 RRF(Reciprocal Rank Fusion)가 가장 간단합니다.

## 7단계 · 저장과 다시 쓰기

조 · 항 · 호 표를 Parquet 으로 저장합니다(3주차 13장). FTS 색인은 DuckDB 데이터베이스 파일 안에 저장되므로, 파일로 연결해 두면 다음에 색인을 다시 만들 필요가 없습니다.

In [ ]:
law_df.drop(columns=["t_josa", "t_bigram"]).to_parquet("data/m6_pipa_articles.parquet", index=False)

db = duckdb.connect("data/m6_law.duckdb")
db.sql("LOAD fts")
db.sql("CREATE OR REPLACE TABLE law_bigram AS SELECT * FROM 'data/m6_pipa_articles.parquet'")
db.create_function("bigram", bigram, [duckdb.typing.VARCHAR], duckdb.typing.VARCHAR)
db.sql("ALTER TABLE law_bigram ADD COLUMN body VARCHAR")
db.sql("UPDATE law_bigram SET body = bigram(text)")
db.sql("PRAGMA create_fts_index('law_bigram', 'id', 'body', stemmer='none', stopwords='none', ignore='(\\.|[^a-z0-9가-힣])+', overwrite=1)")
db.close()

db = duckdb.connect("data/m6_law.duckdb")                    # 다시 열어도 색인이 남아 있다
db.sql("LOAD fts")
top = db.execute("""
    SELECT article, hang, ho FROM (
        SELECT *, fts_main_law_bigram.match_bm25(id, ?) AS s FROM law_bigram
    ) WHERE s IS NOT NULL ORDER BY s DESC LIMIT 1
""", [bigram("가명정보를 처리할 수 있는 목적은?")]).fetchall()
print(top)
db.close()

[('제28조의2', 1, '')]


> **확인 포인트** — 다시 연 데이터베이스에서 가명정보 질문의 1위가 `제28조의2` 제1항으로 나옵니다.

In [ ]:
assert top[0][0] == "제28조의2"

> **자주 겪는 오류** — 색인은 **만든 순간의 테이블**로 만들어집니다. 뒤에 줄을 넣거나 고치면 색인에는 반영되지 않으므로 `overwrite=1` 로 다시 만드세요.

## 8단계 · 더 해 보기

- 조사 떼기 목록에 「되면 · 된 · 하는 · 하여야」 같은 어미를 넣으면 「유출」 줄 수와 hit@1 이 어떻게 달라지나요? 망가지는 낱말(「사람」 → 「사」?)도 함께 찾아보세요.
- `pip install kiwipiepy` 로 형태소 분석기 Kiwi 를 설치해 명사만 뽑아 색인해 보고, 6단계 표에 한 줄을 더하세요.
- 6단계 질문에 「제17조」처럼 번호를 넣은 질문이 오면 정규식으로 번호를 떼어 `WHERE article = …` 로 먼저 찾고, 나머지만 BM25 로 찾는 함수를 만들어 보세요.
- P4 의 임베딩 검색 결과와 BM25 2-gram 순위를 RRF 로 합쳐 15문항 정답률을 다시 재 보세요.